# 01 Preprocessing
## 전처리


In [1]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
SAMPLE_DIR = PROJECT_ROOT / "data" / "sample"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

def load_table(name):
    return pd.read_csv(TABLE_DIR / name)

def save_figure(name):
    path = FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    return path

def pct(x, digits=2):
    return (x.astype(float) * 100).round(digits)


## Monthly Records
### 월별 고객 이력


In [2]:
raw_shape = pd.DataFrame(
    [
        ["train.parquet", "customer-month", 5_531_451, 190],
        ["test.parquet", "customer-month", 11_363_762, 190],
        ["train_labels.csv", "customer", 458_913, 2],
    ],
    columns=["table", "unit", "rows", "columns"],
)
display(raw_shape)

customer_month = pd.DataFrame(
    {
        "customer_ID": ["C001"] * 5 + ["C002"] * 5 + ["C003"] * 5,
        "S_2": pd.to_datetime(
            ["2017-11-30", "2017-12-31", "2018-01-31", "2018-02-28", "2018-03-31"] * 3
        ),
        "B_1": [0.18, 0.20, 0.22, 0.28, 0.35, 0.05, 0.04, 0.06, 0.06, 0.07, 0.44, 0.50, 0.57, 0.63, 0.72],
        "D_39": [0, 0, 1, 1, 2, 0, 0, 0, 0, 1, 2, 3, 4, 4, 5],
        "S_3": [0.12, 0.11, 0.14, 0.16, 0.18, 0.03, np.nan, 0.04, 0.04, 0.05, 0.21, 0.25, 0.28, 0.31, 0.33],
        "D_63": ["CO", "CO", "CO", "CO", "CO", "CR", "CR", "CR", "CR", "CR", "CO", "CO", "XL", "XL", "XL"],
        "target": [0] * 5 + [0] * 5 + [1] * 5,
    }
)
customer_month["statement_order"] = customer_month.groupby("customer_ID").cumcount() + 1
display(customer_month)


,table,unit,rows,columns
0,train.parquet,customer-month,5531451,190
1,test.parquet,customer-month,11363762,190
2,train_labels.csv,customer,458913,2


,customer_ID,S_2,B_1,D_39,S_3,D_63,target,statement_order
0,C001,2017-11-30,0.18,0,0.12,CO,0,1
1,C001,2017-12-31,0.20,0,0.11,CO,0,2
2,C001,2018-01-31,0.22,1,0.14,CO,0,3
3,C001,2018-02-28,0.28,1,0.16,CO,0,4
4,C001,2018-03-31,0.35,2,0.18,CO,0,5
5,C002,2017-11-30,0.05,0,0.03,CR,0,1
6,C002,2017-12-31,0.04,0,NaN,CR,0,2
7,C002,2018-01-31,0.06,0,0.04,CR,0,3
8,C002,2018-02-28,0.06,0,0.04,CR,0,4
9,C002,2018-03-31,0.07,1,0.05,CR,0,5


월별 이력을 고객 단위로 압축한다.


## Feature Layers
### 변수 레이어


In [3]:
numeric_cols = ["B_1", "D_39", "S_3"]
categorical_cols = ["D_63"]

def build_base_features(frame, customer_col, numeric_cols):
    sorted_frame = frame.sort_values([customer_col, "S_2"])
    agg = sorted_frame.groupby(customer_col)[numeric_cols].agg(["last", "first", "mean", "std", "min", "max", "sum", "median", "count"])
    agg.columns = [f"{col}_{stat}" for col, stat in agg.columns]
    return agg

def build_change_features(base, numeric_cols):
    out = pd.DataFrame(index=base.index)
    for col in numeric_cols:
        out[f"{col}_last_minus_mean"] = base[f"{col}_last"] - base[f"{col}_mean"]
        out[f"{col}_last_minus_first"] = base[f"{col}_last"] - base[f"{col}_first"]
        out[f"{col}_last_div_mean"] = base[f"{col}_last"] / base[f"{col}_mean"].replace(0, np.nan)
        out[f"{col}_last_div_first"] = base[f"{col}_last"] / base[f"{col}_first"].replace(0, np.nan)
    return out

def build_temporal_features(frame, customer_col, numeric_cols):
    sorted_frame = frame.sort_values([customer_col, "S_2"])
    grouped = sorted_frame.groupby(customer_col)
    out = pd.DataFrame(index=grouped.size().index)
    for col in numeric_cols:
        out[f"{col}_last_minus_lag1"] = grouped[col].last() - grouped[col].nth(-2)
        out[f"{col}_last_minus_lag3"] = grouped[col].last() - grouped[col].nth(-4)
        out[f"{col}_recent_3m_mean"] = grouped[col].tail(3).groupby(sorted_frame[customer_col]).mean()
        out[f"{col}_recent_3m_std"] = grouped[col].tail(3).groupby(sorted_frame[customer_col]).std()
    return out

def build_missing_features(frame, customer_col, numeric_cols):
    missing_rate = frame.groupby(customer_col)[numeric_cols].apply(lambda x: x.isna().mean())
    missing_rate.columns = [f"{col}_missing_rate" for col in missing_rate.columns]
    missing_count = frame.groupby(customer_col)[numeric_cols].apply(lambda x: x.isna().sum().sum()).to_frame("missing_count_total")
    return missing_rate.join(missing_count)

def build_categorical_features(frame, customer_col, categorical_cols):
    sorted_frame = frame.sort_values([customer_col, "S_2"])
    grouped = sorted_frame.groupby(customer_col)
    out = pd.DataFrame(index=grouped.size().index)
    for col in categorical_cols:
        out[f"{col}_last"] = grouped[col].last()
        out[f"{col}_first"] = grouped[col].first()
        out[f"{col}_nunique"] = grouped[col].nunique()
    return pd.get_dummies(out, dummy_na=True)

feature_layers = pd.DataFrame(
    {
        "layer": ["base", "change", "temporal", "missing", "categorical"],
        "example": [
            "B_1_last, B_1_mean, B_1_std",
            "B_1_last_minus_mean, B_1_last_div_first",
            "B_1_last_minus_lag1, B_1_recent_3m_mean",
            "S_3_missing_rate, missing_count_total",
            "D_63_last_CO, D_63_nunique",
        ],
    }
)
display(feature_layers)


,layer,example
0,base,"B_1_last, B_1_mean, B_1_std"
1,change,"B_1_last_minus_mean, B_1_last_div_first"
2,temporal,"B_1_last_minus_lag1, B_1_recent_3m_mean"
3,missing,"S_3_missing_rate, missing_count_total"
4,categorical,"D_63_last_CO, D_63_nunique"


수준, 변화, 최근성, 결측, 상태 변수를 분리해서 만든다.


## Customer-Level Matrix
### 고객 단위 행렬


In [4]:
base = build_base_features(customer_month, "customer_ID", numeric_cols)
change = build_change_features(base, numeric_cols)
temporal = build_temporal_features(customer_month, "customer_ID", numeric_cols)
missing = build_missing_features(customer_month, "customer_ID", numeric_cols)
categorical = build_categorical_features(customer_month, "customer_ID", categorical_cols)
target = customer_month.groupby("customer_ID")["target"].last().rename("target")

model_table = pd.concat([base, change, temporal, missing, categorical, target], axis=1).reset_index()
X = model_table.drop(columns=["customer_ID", "target"])
y = model_table["target"].astype(int)

matrix_summary = pd.DataFrame(
    {
        "metric": ["customers", "features", "default_rate", "missing_cells"],
        "value": [len(model_table), X.shape[1], round(y.mean(), 4), int(X.isna().sum().sum())],
    }
)
display(matrix_summary)
display(model_table)


,metric,value
0,customers,3.0000
1,features,63.0000
2,default_rate,0.3333
3,missing_cells,20.0000


,customer_ID,B_1_last,B_1_first,B_1_mean,B_1_std,B_1_min,B_1_max,B_1_sum,B_1_median,B_1_count,D_39_last,D_39_first,D_39_mean,D_39_std,D_39_min,D_39_max,D_39_sum,D_39_median,D_39_count,S_3_last,S_3_first,S_3_mean,S_3_std,S_3_min,S_3_max,S_3_sum,S_3_median,S_3_count,B_1_last_minus_mean,B_1_last_minus_first,B_1_last_div_mean,B_1_last_div_first,D_39_last_minus_mean,D_39_last_minus_first,D_39_last_div_mean,D_39_last_div_first,S_3_last_minus_mean,S_3_last_minus_first,S_3_last_div_mean,S_3_last_div_first,B_1_last_minus_lag1,B_1_last_minus_lag3,B_1_recent_3m_mean,B_1_recent_3m_std,D_39_last_minus_lag1,D_39_last_minus_lag3,D_39_recent_3m_mean,D_39_recent_3m_std,S_3_last_minus_lag1,S_3_last_minus_lag3,S_3_recent_3m_mean,S_3_recent_3m_std,B_1_missing_rate,D_39_missing_rate,S_3_missing_rate,missing_count_total,D_63_nunique,D_63_last_CO,D_63_last_CR,D_63_last_XL,D_63_last_nan,D_63_first_CO,D_63_first_CR,D_63_first_nan,target
0,C001,0.35,0.18,0.246,0.069138,0.18,0.35,1.23,0.22,5,2,0,0.8,0.836660,0,2,4,1.0,5,0.18,0.12,0.142,0.028636,0.11,0.18,0.71,0.14,5,0.104,0.17,1.422764,1.944444,1.2,2,2.500000,NaN,0.038,0.06,1.267606,1.500000,NaN,NaN,0.283333,0.065064,NaN,NaN,1.333333,0.57735,NaN,NaN,0.160000,0.020000,0.0,0.0,0.0,0,1,True,False,False,False,True,False,False,0
1,C002,0.07,0.05,0.056,0.011402,0.04,0.07,0.28,0.06,5,1,0,0.2,0.447214,0,1,1,0.0,5,0.05,0.03,0.040,0.008165,0.03,0.05,0.16,0.04,4,0.014,0.02,1.250000,1.400000,0.8,1,5.000000,NaN,0.010,0.02,1.250000,1.666667,NaN,NaN,0.063333,0.005774,NaN,NaN,0.333333,0.57735,NaN,NaN,0.043333,0.005774,0.0,0.0,0.2,1,1,False,True,False,False,False,True,False,0
2,C003,0.72,0.44,0.572,0.109407,0.44,0.72,2.86,0.57,5,5,2,3.6,1.140175,2,5,18,4.0,5,0.33,0.21,0.276,0.047749,0.21,0.33,1.38,0.28,5,0.148,0.28,1.258741,1.636364,1.4,3,1.388889,2.5,0.054,0.12,1.195652,1.571429,NaN,NaN,0.640000,0.075498,NaN,NaN,4.333333,0.57735,NaN,NaN,0.306667,0.025166,0.0,0.0,0.0,0,2,False,False,True,False,True,False,False,1


모델 입력은 customer-level matrix다.
